# 📊 Tech Challenge Fase 1 — Case NPS Preditivo
## Pós-Graduação em Inteligência Artificial para Cientistas de Dados | FIAP

**Objetivo:** Explorar dados operacionais de um e-commerce para identificar quais fatores influenciam a satisfação do cliente (NPS) e propor ações proativas de melhoria.

**Autores:** [Inserir nomes do grupo]  
**Data:** Maio de 2026

---

## 1. Configuração do Ambiente

In [ ]:
# Importação de bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

# Configurações globais de visualização
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.figsize': (10, 6)
})

# Cores padrão para as categorias NPS
COR_DETRATOR  = '#E74C3C'
COR_NEUTRO    = '#F39C12'
COR_PROMOTOR  = '#27AE60'

print("✅ Ambiente configurado com sucesso!")

## 2. Carregamento e Inspeção dos Dados

In [ ]:
# Carregamento da base de dados
df = pd.read_csv('data/desafio_nps_fase_1.csv')

# Dimensões do dataset
print(f"📋 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"\n📌 Colunas disponíveis:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col} ({df[col].dtype})")

# Primeiras linhas para visão geral
print("\n📊 Primeiras 5 linhas:")
df.head()

In [ ]:
# Verificação de qualidade dos dados
print("=" * 50)
print("VERIFICAÇÃO DE QUALIDADE DOS DADOS")
print("=" * 50)

# Valores nulos
nulos = df.isnull().sum()
print(f"\n🔍 Valores nulos por coluna:")
if nulos.sum() == 0:
    print("   ✅ Nenhum valor nulo encontrado!")
else:
    print(nulos[nulos > 0])

# Registros duplicados
duplicatas = df.duplicated().sum()
print(f"\n🔍 Registros duplicados: {duplicatas}")

# Estatísticas descritivas
print("\n📈 Estatísticas descritivas:")
df.describe().round(2)

## 3. Entendimento do Negócio

### 3.1 Qual problema de negócio está sendo resolvido?

O e-commerce em questão enfrenta uma **alta variabilidade no Net Promoter Score (NPS)** entre diferentes perfis de consumidores. Mesmo com indicadores operacionais aparentemente semelhantes, alguns clientes se tornam promotores enquanto outros se tornam detratores.

**Pergunta central:** *Quais fatores operacionais realmente influenciam a satisfação do cliente e como a empresa pode agir de forma proativa para melhorar a experiência antes da aplicação da pesquisa de NPS?*

Atualmente, o NPS é coletado **apenas após** o encerramento da jornada de compra, limitando ações preventivas. A necessidade é transformar dados operacionais em **insights acionáveis**.

### 3.2 Por que o NPS é importante para um e-commerce?

- **Indicador de lealdade:** Mede a disposição do cliente em recomendar a empresa
- **Preditor de crescimento:** Empresas com NPS alto crescem mais rápido
- **Simplicidade e comparabilidade:** Permite benchmarking com o mercado
- **Sinal antecipado de churn:** Detratores têm maior probabilidade de não recomprar
- **Guia de investimentos:** Identifica onde melhorar para maximizar retorno

### 3.3 Áreas que se beneficiam dos insights

| Área | Benefício |
|------|-----------|
| **Logística** | Otimização de prazos e redução de atrasos |
| **Atendimento (SAC)** | Melhoria na resolução e tempo de resposta |
| **Produto** | Identificação de pontos de fricção na jornada |
| **Pricing** | Ajuste de políticas de frete e desconto |
| **Estratégia** | Priorização de ações com maior impacto |

### 3.4 Reflexão: Impacto do NPS

| Dimensão | Impacto |
|----------|---------|
| **Recompra** | Promotores recompram significativamente mais que Detratores |
| **Boca a boca** | Promotores são embaixadores; Detratores amplificam experiências negativas em redes sociais e Reclame Aqui |
| **Market Share** | NPS elevado = maior retenção, maior LTV e vantagem competitiva sustentável |

### 3.5 Indicadores de mercado complementares

- **Benchmarks de NPS do setor:** NPS médio do e-commerce BR varia entre 30 e 50
- **SLA Logístico:** % de entregas dentro do prazo
- **CSAT:** medição pontual de satisfação por interação
- **CES (Customer Effort Score):** esforço do cliente para resolver problemas
- **Reputação externa:** Reclame Aqui, Google Reviews, NPS de concorrentes

## 4. Definição da Target

### 4.1 Qual variável representa a satisfação do cliente?

A variável-alvo é o **`nps_score`**, nota de satisfação do cliente (0 a 10), classificada em:

| Categoria | Faixa | Significado |
|-----------|-------|-------------|
| **Detrator** | 0 a 6 | Insatisfeito, pode prejudicar a marca |
| **Neutro** | 7 a 8 | Satisfeito, mas sem engajamento ativo |
| **Promotor** | 9 a 10 | Leal, recomenda a marca ativamente |

### 4.2 Por que foi escolhida?

- Métrica mais consagrada globalmente para lealdade
- Captura a experiência holística do cliente
- Correlação direta com recompra, recomendação e retenção
- Classificação acionável em 3 grupos

### 4.3 Momento da jornada

Coletada **após o encerramento da jornada de compra** — reflete toda a experiência, da compra ao pós-venda.

### 4.4 Riscos de uso inadequado

- ⚠️ **Viés de seleção:** respondentes podem não representar toda a base
- ⚠️ **Viés temporal:** influenciado por fatores externos momentâneos
- ⚠️ **Subjetividade:** interpretação da escala varia entre perfis
- ⚠️ **Métrica reativa:** coletada após a jornada, limita ação preventiva
- ⚠️ **Data leakage:** `repeat_purchase_30d` e `csat_internal_score` podem ser consequências, não causas

## 5. Preparação dos Dados

In [ ]:
# Criação das categorias NPS conforme classificação padrão
def classificar_nps(nota):
    """Classifica a nota NPS em Detrator, Neutro ou Promotor."""
    if nota <= 6:
        return 'Detrator'
    elif nota <= 8:
        return 'Neutro'
    else:
        return 'Promotor'

df['nps_categoria'] = df['nps_score'].apply(classificar_nps)

# Ordem das categorias para visualizações
CATEGORIAS_ORDEM = ['Detrator', 'Neutro', 'Promotor']

# ── Cálculo do NPS Geral ────────────────────────────────────────
total = len(df)
pct_promotor = (df['nps_categoria'] == 'Promotor').sum() / total * 100
pct_neutro   = (df['nps_categoria'] == 'Neutro').sum() / total * 100
pct_detrator = (df['nps_categoria'] == 'Detrator').sum() / total * 100
nps_geral    = pct_promotor - pct_detrator

print("=" * 55)
print("📊 MÉTRICAS GERAIS DO NPS")
print("=" * 55)
print(f"   Total de registros:  {total}")
print(f"   NPS Médio (nota):    {df['nps_score'].mean():.2f}")
print(f"   ")
print(f"   🟢 Promotores (9-10): {pct_promotor:.1f}% ({(df['nps_categoria']=='Promotor').sum()})")
print(f"   🟡 Neutros    (7-8):  {pct_neutro:.1f}% ({(df['nps_categoria']=='Neutro').sum()})")
print(f"   🔴 Detratores (0-6):  {pct_detrator:.1f}% ({(df['nps_categoria']=='Detrator').sum()})")
print(f"   ")
print(f"   ► NPS GERAL = {nps_geral:.1f}")
print("=" * 55)

## 6. Análise Exploratória dos Dados (EDA)

> *"Imagine que você está explicando isso para um(a) gerente de operações que não entende estatística."*

A seguir, apresentamos a análise com foco em **negócio**, utilizando linguagem acessível e visualizações claras.

### 6.1 Distribuição do NPS

In [ ]:
# Visualização da distribuição das notas NPS e categorias
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Gráfico de barras: distribuição das notas ──────────────────
ax1 = axes[0]
contagem_notas = df['nps_score'].value_counts().sort_index()
cores_notas = [COR_DETRATOR if s <= 6 else COR_NEUTRO if s <= 8 else COR_PROMOTOR 
               for s in contagem_notas.index]

barras = ax1.bar(contagem_notas.index, contagem_notas.values, 
                 color=cores_notas, edgecolor='white', linewidth=0.8)

# Adicionar rótulos nas barras
for barra, valor in zip(barras, contagem_notas.values):
    ax1.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 3,
             str(valor), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax1.set_xlabel('Nota NPS')
ax1.set_ylabel('Quantidade de Clientes')
ax1.set_title('Distribuição das Notas NPS', fontweight='bold')
ax1.set_xticks(range(0, 11))

# Legenda
legendas = [mpatches.Patch(color=COR_DETRATOR, label='Detrator (0-6)'),
            mpatches.Patch(color=COR_NEUTRO, label='Neutro (7-8)'),
            mpatches.Patch(color=COR_PROMOTOR, label='Promotor (9-10)')]
ax1.legend(handles=legendas, loc='upper left')

# ── Gráfico de pizza: categorias NPS ──────────────────────────
ax2 = axes[1]
contagem_cat = df['nps_categoria'].value_counts().reindex(CATEGORIAS_ORDEM)
cores_cat = [COR_DETRATOR, COR_NEUTRO, COR_PROMOTOR]

wedges, texts, autotexts = ax2.pie(
    contagem_cat, labels=CATEGORIAS_ORDEM, colors=cores_cat,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_fontweight('bold')
    t.set_fontsize(13)

ax2.set_title(f'Categorias NPS — NPS Geral = {nps_geral:.1f}', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/fig1_distribuicao_nps.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Correlação com o NPS

In [ ]:
# Variáveis numéricas para análise de correlação
variaveis_numericas = [
    'customer_age', 'customer_tenure_months', 'order_value', 'items_quantity',
    'discount_value', 'payment_installments', 'delivery_time_days',
    'delivery_delay_days', 'freight_value', 'delivery_attempts',
    'customer_service_contacts', 'resolution_time_days',
    'complaints_count', 'repeat_purchase_30d', 'csat_internal_score'
]

# Cálculo das correlações com nps_score
correlacoes = df[variaveis_numericas + ['nps_score']].corr()['nps_score'].drop('nps_score').sort_values()

print("=" * 55)
print("📈 CORRELAÇÃO DE CADA VARIÁVEL COM nps_score")
print("=" * 55)
for var, val in correlacoes.items():
    sinal = "🔴" if val < -0.05 else "🟢" if val > 0.05 else "⚪"
    print(f"   {sinal} {var:>30s}: {val:+.3f}")

# ── Mapa de calor ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))

# Renomear para português
nomes_pt = {
    'customer_age': 'Idade', 'customer_tenure_months': 'Tempo Cliente',
    'order_value': 'Valor Pedido', 'items_quantity': 'Qtde Itens',
    'discount_value': 'Valor Desconto', 'payment_installments': 'Parcelas',
    'delivery_time_days': 'Tempo Entrega', 'delivery_delay_days': 'Atraso Entrega',
    'freight_value': 'Valor Frete', 'delivery_attempts': 'Tentativas Entrega',
    'customer_service_contacts': 'Contatos SAC', 'resolution_time_days': 'Tempo Resolução',
    'complaints_count': 'Reclamações', 'repeat_purchase_30d': 'Recompra 30d',
    'csat_internal_score': 'CSAT Interno', 'nps_score': 'NPS Score'
}

matriz_corr = df[variaveis_numericas + ['nps_score']].corr()
matriz_corr_pt = matriz_corr.rename(index=nomes_pt, columns=nomes_pt)
mascara = np.triu(np.ones_like(matriz_corr_pt, dtype=bool), k=1)

sns.heatmap(matriz_corr_pt, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mascara, ax=ax, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 9})
ax.set_title('Mapa de Correlação entre Variáveis', fontweight='bold', fontsize=15)

plt.tight_layout()
plt.savefig('reports/fig2_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Fatores Críticos para a Satisfação

Comparando os principais indicadores operacionais entre as categorias NPS, conseguimos visualizar **o que separa clientes satisfeitos dos insatisfeitos**.

In [ ]:
# Boxplots dos fatores operacionais por categoria NPS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Boxplot: Atraso na Entrega ────────────────────────────────
ax1 = axes[0]
dados_atraso = [df.loc[df['nps_categoria'] == cat, 'delivery_delay_days'].values 
                for cat in CATEGORIAS_ORDEM]
bp1 = ax1.boxplot(dados_atraso, labels=CATEGORIAS_ORDEM, patch_artist=True,
                  medianprops=dict(color='black', linewidth=2))
for patch, cor in zip(bp1['boxes'], [COR_DETRATOR, COR_NEUTRO, COR_PROMOTOR]):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
ax1.set_ylabel('Dias de Atraso')
ax1.set_title('Atraso na Entrega por Categoria NPS', fontweight='bold')

# ── Boxplot: Reclamações ──────────────────────────────────────
ax2 = axes[1]
dados_reclam = [df.loc[df['nps_categoria'] == cat, 'complaints_count'].values 
                for cat in CATEGORIAS_ORDEM]
bp2 = ax2.boxplot(dados_reclam, labels=CATEGORIAS_ORDEM, patch_artist=True,
                  medianprops=dict(color='black', linewidth=2))
for patch, cor in zip(bp2['boxes'], [COR_DETRATOR, COR_NEUTRO, COR_PROMOTOR]):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
ax2.set_ylabel('Número de Reclamações')
ax2.set_title('Reclamações por Categoria NPS', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/fig3_boxplots_fatores.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Ponto de Ruptura — Atraso na Entrega

> *"A partir de quantos dias de atraso o cliente fica realmente insatisfeito?"*

Essa é uma das perguntas mais importantes para a operação. Abaixo, identificamos o **limiar crítico**.

In [ ]:
# Criação de faixas de atraso para identificar ponto de ruptura
faixas_atraso = [-1, 0, 2, 5, 10, 100]
rotulos_atraso = ['Sem atraso', '1–2 dias', '3–5 dias', '6–10 dias', '10+ dias']
df['faixa_atraso'] = pd.cut(df['delivery_delay_days'], bins=faixas_atraso, labels=rotulos_atraso)

# Cálculo do NPS médio e contagem por faixa
nps_por_atraso = df.groupby('faixa_atraso', observed=False)['nps_score'].mean()
contagem_atraso = df.groupby('faixa_atraso', observed=False).size()

# Visualização
fig, ax = plt.subplots(figsize=(10, 5))
cores_atraso = ['#27AE60', '#F1C40F', '#E67E22', '#E74C3C', '#943126']
barras = ax.bar(range(len(nps_por_atraso)), nps_por_atraso.values, 
                color=cores_atraso, edgecolor='white', linewidth=1)

# Rótulos com NPS médio e contagem
for bar, valor, cnt in zip(barras, nps_por_atraso.values, contagem_atraso.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{valor:.1f}\n(n={cnt})', ha='center', va='bottom', 
            fontsize=10, fontweight='bold')

ax.set_xticks(range(len(rotulos_atraso)))
ax.set_xticklabels(rotulos_atraso)
ax.set_xlabel('Faixa de Atraso na Entrega')
ax.set_ylabel('NPS Médio')
ax.set_title('⚠️ Ponto de Ruptura: NPS Médio por Faixa de Atraso', fontweight='bold')
ax.axhline(y=df['nps_score'].mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'Média geral: {df["nps_score"].mean():.1f}')
ax.legend()

plt.tight_layout()
plt.savefig('reports/fig4_ponto_ruptura.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 INSIGHT: A satisfação cai significativamente a partir de 3 dias de atraso!")

### 6.5 Impacto dos Contatos com o SAC

In [ ]:
# Agrupamento por faixa de contatos com SAC
df['faixa_contatos_sac'] = df['customer_service_contacts'].apply(
    lambda x: '4+' if x >= 4 else str(x)
)
ordem_sac = ['0', '1', '2', '3', '4+']

# Cálculo do NPS médio e contagem por faixa
nps_por_sac = df.groupby('faixa_contatos_sac')['nps_score'].mean().reindex(ordem_sac)
contagem_sac = df.groupby('faixa_contatos_sac').size().reindex(ordem_sac)

# Visualização
fig, ax = plt.subplots(figsize=(9, 5))
cores_sac = ['#27AE60', '#2ECC71', '#F1C40F', '#E67E22', '#E74C3C']
barras = ax.bar(range(len(nps_por_sac)), nps_por_sac.values, 
                color=cores_sac, edgecolor='white')

for bar, valor, cnt in zip(barras, nps_por_sac.values, contagem_sac.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{valor:.1f}\n(n={cnt})', ha='center', va='bottom', 
            fontsize=10, fontweight='bold')

ax.set_xticks(range(len(ordem_sac)))
ax.set_xticklabels(ordem_sac)
ax.set_xlabel('Número de Contatos com o SAC')
ax.set_ylabel('NPS Médio')
ax.set_title('NPS Médio por Quantidade de Contatos com Atendimento', fontweight='bold')
ax.axhline(y=df['nps_score'].mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'Média geral: {df["nps_score"].mean():.1f}')
ax.legend()

plt.tight_layout()
plt.savefig('reports/fig5_nps_por_sac.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 INSIGHT: Clientes com 4+ contatos com SAC têm NPS significativamente menor!")

### 6.6 Análise Regional

In [ ]:
# NPS médio por região geográfica
nps_por_regiao = df.groupby('customer_region')['nps_score'].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.barh(nps_por_regiao.index, nps_por_regiao.values, 
                 color='#3498DB', edgecolor='white')

for bar, valor in zip(barras, nps_por_regiao.values):
    ax.text(valor + 0.05, bar.get_y() + bar.get_height()/2, f'{valor:.2f}',
            ha='left', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('NPS Médio')
ax.set_ylabel('Região')
ax.set_title('NPS Médio por Região do Cliente', fontweight='bold')
ax.set_xlim(0, max(nps_por_regiao.values) + 0.8)

plt.tight_layout()
plt.savefig('reports/fig6_nps_regiao.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 INSIGHT: Diferenças regionais são pequenas — o problema é operacional, não geográfico.")

### 6.7 Impacto na Recompra

> *"Clientes satisfeitos realmente compram de novo?"*

A resposta é um **sim inequívoco**. A taxa de recompra em 30 dias varia dramaticamente entre categorias.

In [ ]:
# Taxa de recompra por categoria NPS
taxa_recompra = df.groupby('nps_categoria')['repeat_purchase_30d'].mean().reindex(CATEGORIAS_ORDEM) * 100

fig, ax = plt.subplots(figsize=(8, 5))
cores_recompra = [COR_DETRATOR, COR_NEUTRO, COR_PROMOTOR]
barras = ax.bar(CATEGORIAS_ORDEM, taxa_recompra.values, color=cores_recompra, 
                edgecolor='white', linewidth=1)

for bar, valor in zip(barras, taxa_recompra.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{valor:.1f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Taxa de Recompra em 30 dias (%)')
ax.set_title('Taxa de Recompra por Categoria NPS', fontweight='bold')
ax.set_ylim(0, max(taxa_recompra.values) + 10)

plt.tight_layout()
plt.savefig('reports/fig7_recompra.png', dpi=150, bbox_inches='tight')
plt.show()

# Cálculo do multiplicador
mult = taxa_recompra['Promotor'] / taxa_recompra['Detrator']
print(f"\n💡 INSIGHT: Promotores recompram {mult:.1f}x mais que Detratores!")
print(f"   Promotor: {taxa_recompra['Promotor']:.1f}%")
print(f"   Neutro:   {taxa_recompra['Neutro']:.1f}%")
print(f"   Detrator: {taxa_recompra['Detrator']:.1f}%")

### 6.8 Resumo: Médias por Categoria NPS

In [ ]:
# Tabela comparativa de médias por categoria NPS
variaveis_chave = [
    'delivery_delay_days', 'complaints_count', 'customer_service_contacts',
    'resolution_time_days', 'delivery_attempts', 'csat_internal_score', 
    'repeat_purchase_30d'
]

# Nomes amigáveis para a tabela
nomes_amigaveis = {
    'delivery_delay_days': 'Atraso Entrega (dias)',
    'complaints_count': 'Reclamações (média)',
    'customer_service_contacts': 'Contatos SAC (média)',
    'resolution_time_days': 'Tempo Resolução (dias)',
    'delivery_attempts': 'Tentativas Entrega',
    'csat_internal_score': 'CSAT Interno',
    'repeat_purchase_30d': 'Taxa Recompra 30d'
}

tabela_medias = df.groupby('nps_categoria')[variaveis_chave].mean().reindex(CATEGORIAS_ORDEM)
tabela_medias = tabela_medias.rename(columns=nomes_amigaveis).round(2)

print("=" * 75)
print("📊 MÉDIAS DOS INDICADORES-CHAVE POR CATEGORIA NPS")
print("=" * 75)
print(tabela_medias.T.to_string())
print("=" * 75)
print("\n💡 CONCLUSÃO: Detratores enfrentam mais atraso, mais reclamações,")
print("   mais contatos com SAC e recompram significativamente menos.")

## 7. Conclusões e Recomendações

### 🔑 Principais Descobertas

1. **Atraso na entrega é o principal destruidor de NPS** — correlação mais forte entre todas as variáveis operacionais
2. **Ponto de ruptura identificado: 3 dias de atraso** — após esse limiar, a satisfação despenca
3. **Múltiplos contatos com SAC = insatisfação** — clientes que ligam 4+ vezes têm NPS muito inferior
4. **Promotores recompram 4,3x mais que Detratores** — impacto financeiro direto e mensurável
5. **Diferenças regionais são mínimas** — o problema é operacional, não geográfico

### ✅ 5 Recomendações Estratégicas

| # | Recomendação | Área | Impacto Esperado |
|---|--------------|------|------------------|
| 1 | **Zerar atrasos >3 dias** — alertas automáticos, contingência regional | Logística | 🔴 Crítico |
| 2 | **Reduzir tentativas de entrega** — rastreamento real-time, agendamento flexível | Logística | 🟠 Alto |
| 3 | **Acelerar resolução no SAC** — FCR, autonomia aos atendentes, chatbot | Atendimento | 🟠 Alto |
| 4 | **Programa de recuperação de Detratores** — contato proativo, compensações | CX | 🟡 Médio |
| 5 | **Converter Neutros em Promotores** — fidelidade, comunicação personalizada | Marketing | 🟡 Médio |

### ⚠️ Limitações

- Correlação não é causalidade — recomenda-se validar com **testes A/B**
- Viés de resposta — clientes silenciosos podem ter perfil diferente
- Dados transversais — tendências e sazonalidades não foram avaliadas
- Variáveis latentes (qualidade do produto, UX digital) não estão nos dados
- `repeat_purchase_30d` e `csat_internal_score` podem ter **data leakage** em modelos preditivos

---

*Relatório gerado em Maio de 2026 | Tech Challenge Fase 1 — FIAP*